# Stimulus generation
This notebook generates a reVISit config JSON from a high-level ISSUES list, automatically creating the 6 congruence patterns (choose 2 of 4) and wrapping them in a latinSquare block.

In [2]:
import revisitpy as rvt
from itertools import combinations
from pathlib import Path

/opt/miniconda3/envs/critique/lib/python3.13/site-packages/revisitpy/models.py:5914: UserWarning: Field name "schema" in "StudyConfig" shadows an attribute in parent "BaseModel"
  class StudyConfig(BaseModel):


## Global Vars

(do high level edits here)

In [3]:
ISSUES = [
    dict(
        issueId="gun",
        proposition="Gun ownership should be more strictly regulated in the United States to reduce violent crime.",
        proImagePath="my-study/stimuli/gun_pro.png",
        conImagePath="my-study/stimuli/gun_con.png",
    ),
    dict(
        issueId="mil",
        proposition="The United States Government should increase its military budget to protect national security.",
        proImagePath="my-study/stimuli/mil_pro.png",
        conImagePath="my-study/stimuli/mil_con.png",
    ),
    dict(
        issueId="trump",
        proposition="I approve of the way Donald Trump is handling his job as President.",
        proImagePath="my-study/stimuli/trump_pro.png",
        conImagePath="my-study/stimuli/trump_con.png",
    ),
    dict(
        issueId="immig",
        proposition="Immigration in the United States should be more strictly regulated to reduce the number of undocumented immigrants.",
        proImagePath="my-study/stimuli/immig_pro.png",
        conImagePath="my-study/stimuli/immig_con.png",
    ),
]

PHASES = [
    "stimulus-view-chart",
    "stimulus-pre-words",
    "stimulus-getready",
    "stimulus-record",
    "stimulus-post-words",
]

TRIALS_FUNCTION_PATH = "my-study/trials.ts"

ISSUE_IDS = [x["issueId"] for x in ISSUES]
CONGRUENCE_PATTERNS = list(combinations(ISSUE_IDS, 2))  # 6 patterns
CONGRUENCE_PATTERNS

[('gun', 'mil'),
 ('gun', 'trump'),
 ('gun', 'immig'),
 ('mil', 'trump'),
 ('mil', 'immig'),
 ('trump', 'immig')]

## Generating Components

In [4]:
# Belief matrix (your existing one)
belief_matrix = rvt.response(
    id="belief-matrix",
    type="matrix-radio",
    prompt="Please rate how much you agree or disagree with the following statements:",
    answerOptions=["Strongly Disagree", "Disagree", "Neutral", "Agree", "Strongly Agree"],
    questionOptions=[x["proposition"] for x in ISSUES],
    required=True,
)

belief_questionnaire = rvt.component(
    component_name__="belief-questionnaire",
    type="questionnaire",
    recordAudio=False,
    response=[belief_matrix],
)

# --- Think-aloud phase pages (your 5 reusable components) ---
stimulus_view_chart = rvt.component(
    component_name__="stimulus-view-chart",
    type="react-component",
    path="my-study/assets/PngChartPhase.tsx",
    nextButtonEnableTime=30000,
    recordAudio=False,
    parameters={"imagePath": None},
    response=[],
)

stimulus_pre_words = rvt.component(
    component_name__="stimulus-pre-words",
    type="react-component",
    path="my-study/assets/PngChartPhase.tsx",
    recordAudio=False,
    parameters={"imagePath": None},
    response=[
        rvt.response(
            id="pre_words",
            type="shortText",
            location="belowStimulus",
            prompt="Before critiquing: describe the chart in 3–5 words.",
            required=True,
            placeholder="3–5 words",
        )
    ],
)

stimulus_getready = rvt.component(
    component_name__="stimulus-getready",
    type="markdown",
    path="my-study/assets/ready_to_record.md",
    recordAudio=False,
    nextButtonText="Start recording",
    response=[],
)

stimulus_record = rvt.component(
    component_name__="stimulus-record",
    type="react-component",
    path="my-study/assets/PngChartPhase.tsx",
    nextButtonEnableTime=60000,
    recordAudio=True,
    parameters={"imagePath": None},
    response=[],
)

stimulus_post_words = rvt.component(
    component_name__="stimulus-post-words",
    type="react-component",
    path="my-study/assets/PngChartPhase.tsx",
    recordAudio=False,
    parameters={"imagePath": None},
    response=[
        rvt.response(
            id="post_words",
            type="shortText",
            location="belowStimulus",
            prompt="After critiquing: describe the chart in 3–5 words.",
            required=True,
            placeholder="3–5 words",
        )
    ],
)

print(stimulus_view_chart)

{
    "nextButtonEnableTime": 30000.0,
    "parameters": {
        "imagePath": null
    },
    "path": "my-study/assets/PngChartPhase.tsx",
    "recordAudio": false,
    "response": [],
    "type": "react-component"
}


## Build dynamic blocks from components

In [ ]:
dynamic_blocks = []
for (a, b) in CONGRUENCE_PATTERNS:
    dyn = rvt.sequence(
        id=f"thinkaloud_{a}_{b}",
        order="fixed",
        functionPath=TRIALS_FUNCTION_PATH,
        parameters={
            "issues": ISSUES,
            "phases": PHASES,
            "congruentIssues": [a, b],
        },
    )
    dynamic_blocks.append(dyn)

congruence_assignment = rvt.sequence(
    order="latinSquare",
    numSamples=1,
    components=[dynamic_blocks]
)

print(congruence_assignment)

RevisitError: There was an error. 
----------------------------------------------------

"Type" is required on Component.
